
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 프롬프트를 넘어서 – 검색 에이전트와 컨텍스트 엔지니어링

## 서론

생성형 AI 애플리케이션이 프로토타입에서 본작동으로 넘어가면서 개발자들은 종종 성능 한계에 직면합니다. 아무리 교묘하게 명령어를 다시 써도, 모델이 학습되지 않은 사실을 강제로 알게 하거나, 중요한 데이터가 없을 때 그것이 환각을 일으키는 것을 막을 수 없습니다. 이는 query 제작에 집중하는 **프롬프트 엔지니어링**에서 모델에 제공되는 전체 환경을 설계하는 **컨텍스트 엔지니어링**으로의 근본적인 전환을 의미합니다. 이번 강의에서는 프롬프트의 경계를 정의하고, **Retrieval Augmented Generation(RAG)** 의 아키텍처 솔루션을 소개하며, 신뢰성과 비용 효율성을 위한 컨텍스트 윈도우 공학이라는 고급 분야를 살펴보겠습니다.

## 수업 목표

- 범위와 목표에 따라 프롬프트 엔지니어링과 컨텍스트 엔지니어링을 구분하라
- RAG 아키텍처가 필요한 프롬프트의 세 가지 주요 실패 모드를 식별하라
- RAG 아키텍처의 과제와 이를 해결하기 위한 전략을 설명하라
- 컨텍스트 엔지니어링의 기본 원리와 에이전트 시스템에서의 중요성을 정의하라
- 모델 성능 최적화를 위해 컨텍스트 엔지니어링의 주요 원칙을 적용하라
- 에이전트 시스템의 비용, 지연 시간 및 성능 관리를 위한 전략을 구현하라

**코스 범위에 대한 주의 사항:** 이 수업은 건축 인식을 위한 컨텍스트 엔지니어링 개념을 소개합니다. 실습 연습은 특히 기초 RAG 구현에 중점을 둡니다.

## A. 프롬프트 엔지니어링의 기초

복잡한 AI 아키텍처를 구축하기 전에, 먼저 상호작용의 기본 단위를 숙달해야 합니다: 프롬프트. 이 섹션에서는 모델 행동을 안내하는 지침을 만드는 예술을 탐구하며, 통계적 예측과 진정한 추론 능력을 구분할 것입니다. 하지만 프롬프트를 다듬다 보면, 모델이 훈련 데이터만으로 알 수 있는 한계에 부딪히게 됩니다.

### A1. 프롬프트 엔지니어링의 정의

프롬프트 엔지니어링은 입력 텍스트(프롬프트)를 정제하여 대형 언어 모델(LLM)이 생성하는 출력을 최적화하는 실천입니다. 이것은 교육층에 초점을 맞춘 **전술** 분야입니다. 우리는 **Few-Shot Prompting**(예시 제공)과 **Persona Adoption**(역할 할당)과 같은 기법을 사용하여 모델의 행동과 형식을 안내합니다. 이상적으로는 프롬프트 엔지니어링이 모델을 추론 엔진으로 취급하여 이를 사전 학습된 가중치를 활용해 문제를 효과적으로 해결하도록 유도합니다.

### A2. 추론 vs. 비추론 모델

효과적인 프롬팅은 기본 모델의 역량을 이해하는 것이 필요합니다. 두 가지 주요 범주를 살펴보겠습니다:

- **비추론 모델(예: Llama 3, GPT-4o):** 이 모델들은 통계적 가능도를 바탕으로 다음 토큰을 예측합니다. 복잡한 논리를 해체하고 잘못된 결론에 서두르지 않도록 **Chain of Thought (CoT)** 프롬프트("단계별로 생각하라")와 같은 명확한 안내가 필요합니다.

- **추론 모델(예: OpenAI o1):** 이 모델들은 최종 답변을 도출하기 전에 자체 내부 사고 사슬을 생성하도록 훈련됩니다. 이러한 모델에서 수동 CoT 프롬프트는 종종 중복되거나 역효과를 낼 수 있습니다. 추론 모델에 대한 맥락 엔지니어링은 *사고 과정*보다는 *목표*와 *제약*을 정의하는 데 중점을 둡니다.

### A3. 프롬팅의 경계

프롬프트 엔지니어링에는 엄격한 경계가 있습니다. 아무리 명령어를 다듬어도 모델의 학습 데이터에 내재된 다음과 같은 한계를 극복할 수 없습니다:

1. **지식 기준점:** 모델은 학습 데이터 종료일 이후에 발생한 사건에 대한 질문에 답할 수 없습니다.
   - *예시 프롬프트:* `Who won the 2025 Nobel Prize in Physics?`

1. **환각:** 외부 참고 없이 구체적인 사실을 요구할 때, 모델은 종종 진실보다 그럴듯함을 우선시하며 인용이나 데이터 포인트를 조작합니다.
   - *예시 프롬프트:* `Find a scientific reference proving that avocado reduces blood sugar levels`

1. **모호성:** 사적인 맥락이 없으면, 모델은 기본적으로 일반적인 해석으로 나아갑니다.
   - *예시 프롬프트:* `Explain how to secure a lakehouse.` (이 경우 Databricks Data Lakehouse 거버넌스가 아닌 물리적 주택 보안에 대한 조언을 촉발합니다.)

## B. Retrieval Augmented Generation (RAG)

프롬프트 엔지니어링이 모델의 반응 방식을 최적화하지만, 그것은 모델이 *무엇*을 알고 있는지에 대한 근본적인 문제를 해결할 수 없습니다. 정지된 학습 데이터와 환각의 한계를 극복하기 위해, 우리는 내부 기억에 의존하는 아키텍처 접근법에서 외부 맥락을 활용하는 방식으로 전환해야 합니다. 이 섹션에서는 모델의 사전 학습 지식과 독점 데이터 사이의 간극을 메우는 중요한 프레임워크인 Retrieval Augmented Generation (RAG)을 소개합니다.

**RAG vs. 검색 에이전트:** 
RAG는 특정 도구나 에이전트 논리와 무관하게 검색과 생성을 결합하는 아키텍처 패턴을 의미합니다. **검색 에이전트**는 반면, 이 패턴을 구체적으로 구현하여 실제 시스템 내에서 쿼리 라우팅, 검색 오케스트레이션, 컨텍스트 어셈블리를 처리합니다.

### B1. RAG의 정의

앞서 정의한 지식 경계를 해결하기 위해, 메모리 기반 접근법에서 **Retrieval Augmented Generation (RAG)** 로 알려진 맥락 기반 아키텍처로 전환합니다. RAG는 프롬프트에 독점 또는 실시간 데이터를 주입하여 모델이 내부 메모리가 아닌 제공된 사실에 따라 응답할 수 있도록 합니다.

RAG 과정은 세 가지 키 단계로 구성됩니다:

- **검색:** 시스템은 관련 데이터 청크를 검색하기 위해 지식 베이스(**Databricks AI Search**를 통해 색인됨)
- **증강:** 시스템은 이 청크들을 컨텍스트 창에 주입합니다
- **생성:** 모델은 주입된 데이터만을 사용하여 답변을 합성합니다

### B2. 컨텍스트 과제

RAG가 지식 격차를 해결하는 반면, 이는 새로운 도전 과제를 도입합니다: **맥락 부패**. 초기 RAG 구현은 개발자들이 대량의 문서를 불러 프롬프트에 붙여넣는 경우가 많아 실패하는 경우가 많았습니다. 이 접근법은 모델을 압도하여 다음과 같은 결과를 낳습니다:

- **맥락 중독:** 모델을 혼란스럽게 하는 관련성 없거나 상충되는 정보의 포함
- **중간에 묻히다:** 모델이 긴 컨텍스트 창 중간에 묻혀 있는 정보를 무시하고, 가장 앞부분이나 가장 끝부분의 데이터를 우선시하는 경향

이러한 도전 과제는 전략적 맥락 관리의 필요성을 강조하며, 다음 섹션에서 이를 탐구할 것입니다.

## C. 컨텍스트 엔지니어링의 원리

단순히 데이터를 가져오는 것만으로는 충분하지 않습니다; 프롬프트에 원시 정보를 쏟아붓는 것은 명확성보다는 혼란을 초래하는 경우가 많습니다. 기본 RAG에서 본용 시스템으로 전환함에 따라, 우리는 컨텍스트 윈도우를 수동적인 컨테이너가 아니라 모델 행동을 적극적으로 형성하는 설계된 환경으로 다뤄야 합니다. 이 섹션에서는 컨텍스트 엔지니어링의 원칙을 개괄하며, 신뢰성과 정확성을 보장하기 위해 정보를 구조화, 필터링, 접지 처리하는 방법을 강조하겠습니다.

### C1. 맥락 환경 정의

**컨텍스트 엔지니어링**은 전체 입력 창의 전략적 설계입니다. 이는 단일 명령어 작성을 넘어 전체 시스템 상태를 관리하는 단계로 나아갑니다. 우리는 **시스템 명령어**, **대화 기록**, **검색된 데이터**, **사용자 제약 조건** 간의 상호작용을 조율하여 모델이 작업을 효과적으로 수행하는 데 필요한 신호를 정확히 갖추도록 합니다.

### C2. 시스템 프롬프트 설계

컨텍스트 엔지니어링 패러다임에서 시스템 프롬프트는 단순한 요청이 아니라 모델이 어떻게 작동해야 하는지 정의하는 행동 프로그램입니다.

효과적인 시스템 프롬프트의 주요 구성 요소는 다음과 같습니다:

- **역할 정의:** 페르소나를 명시적으로 정의하세요 (예: "당신은 Databricks 보안 아키텍트입니다")
- **부정적 제약 조건:** 모델이 할 수 없는 것을 정의하세요 (예: "경쟁 제품에 대해 언급하지 말라" 또는 "명시적으로 요청되지 않는 한 코드를 제공하지 마세요")
- **출력 서식:** 구조화된 출력(예: JSON, YAML, Markdown 테이블)을 강제하여 하위 애플리케이션이 응답을 결정론적으로 파싱할 수 있도록 합니다

### C3. 엄격한 그라운딩과 청킹

환각을 막고 정확성을 보장하기 위해 회수는 엄격히 관리되어야 합니다.

주요 전략에는 다음이 포함됩니다:

- **접지 명령어:** 모델을 검색된 문맥에 묶기 위해 명시적 명령어를 사용하세요. 예를 들어: *"제공된 맥락 청크만을 사용하여 답변하세요. 답변이 없다면 '그 정보가 없습니다'라고 말하세요."*
- **메타데이터 필터링:** **Unity Catalog** 메타데이터를 활용해 모델에 도달하기 전에 검색을 필터링합니다. 예를 들어, 사용자가 "2024년 수익"에 대해 묻는다면, 시스템은 모델이 2023년 데이터를 못 보도록 `year=2024`에서 청크를 필터링해야 합니다.

### C4. 다중 턴 상태 관리

에이전트와 같은 긴 대화가 필요한 애플리케이션의 경우, 컨텍스트 창은 결국 채워집니다. 컨텍스트 엔지니어링은 이 상태를 관리하기 위한 전략적 접근이 필요합니다:

- **요약:** 주기적으로 대화 기록을 주요 결정과 사실의 요약으로 압축하세요
- **이동 창:** 가장 오래된 메시지를 버려 토큰 공간을 확보해 새 검색을 위해 사용하세요
- **선택적 지속성:** 어떤 정보(예: 사용자 이름, 현재 프로젝트 ID)가 문맥에 영구적으로 남아 있어야 하는지, 어떤 것은 버려야 하는지 결정합니다

이러한 전략들은 토큰 한도 내에서 관련 맥락을 유지하도록 보장합니다.

## D. 맥락 제약 조건: 토큰 예산과 창 제한

가장 우아하게 설계된 맥락조차도 컴퓨트 리소스와 모델 아키텍처의 엄격한 제약에 따릅니다. 애플리케이션을 확장함에 따라, 포괄적인 맥락에 대한 욕구와 토큰 한도, 지연 시간, 운영 비용 등의 현실 사이에서 균형을 맞춰야 합니다. 이 섹션에서는 컨텍스트 윈도우의 경제학을 살펴보며, 응답 품질을 희생하지 않으면서 토큰 예산을 최적화하는 전략을 제시합니다.

### D1. 컨텍스트 Windows 이해하기

모든 모델에는 **컨텍스트 Windows 제한**(예: 8k, 32k 또는 128k 토큰)이 있습니다. 이는 모델이 사용할 수 있는 작업 기억의 하드 한계를 나타냅니다.

컨텍스트 창은 다음과 같습니다:

- **입력 토큰:** 모델에 보내는 텍스트 (명령어 + 검색된 문서 + 이력)
- **출력 토큰:** 모델이 생성하는 텍스트

**트레이드오프:** 더 많은 데이터를 검색한 것으로 창을 채울수록 모델의 추론 능력은 저하되고('Lost in the Middle' 현상), 지연 시간이 크게 증가합니다. 전략적 맥락 관리는 성능 유지에 필수적입니다.

### D2. 토큰 경제학과 최적화

맥락은 무료가 아닙니다. **Databricks 재단 모델 APIs**(및 기타 제공자)는 소비된 투입량과 산출량 토큰에 따라 요금을 부과합니다.

주요 고려사항:

- **비용 관리:** 쿼리 하나당 50개의 문서를 가져오는 순진한 RAG 시스템은 토큰 예산을 빠르게 소진시킵니다

**최적화 전략:**

- **적시 검색:** 채팅 시작 시 전체 매뉴얼을 로드하는 대신, 사용자가 관련 질문을 할 때만 특정 섹션을 검색할 수 있는 도구를 에이전트에게 제공하세요
- **재순위 조정:** 리랭커 모델을 사용해 상위 50개의 검색된 청크를 점수 부여하고, 최종 컨텍스트 창에 가장 관련성 높은 3-5개 청크만 주입하세요

이러한 전략들은 포괄적인 맥락과 비용 효율성 및 성능을 균형 있게 유지하는 데 도움을 줍니다.

## E. 요약

프롬프트 엔지니어링에서 컨텍스트 엔지니어링으로의 전환은 '마이크로 최적화'에서 '매크로 아키텍처'로의 근본적인 전환을 의미합니다. 프롬프트는 어조와 형식을 제어하지만, LLM에 내재된 지식 격차를 메우지는 못합니다. RAG 아키텍처는 외부 데이터를 주입하여 이를 해결하지만, 컨텍스트 윈도우 관리와 관련된 복잡성을 도입합니다. Context Engineering은 입력을 엄격하게 구조화하고, 기본 규칙을 적용하며, 토큰 예산을 관리함으로써 신뢰할 수 있고 비용 효율적인 AI 시스템을 만듭니다.

**주요 내용:**

1. **검색 에이전트 아키텍처:** 검색 컴포넌트를 사용해 지식 격차를 메우고, 컨텍스트 엔지니어링을 적용해 검색 에이전트의 성능, 신뢰성, 비용 최적화를 합니다.
2. **맥락은 유한하다:** 맥락 창을 예산처럼 관리하세요. 필터링과 재랭킹을 사용해 모든 토큰의 가치를 극대화하세요.
3. **접지는 필수입니다:** 모델에 *오직* 검색된 데이터만 사용하도록 엄격히 지시하고, **Unity Catalog** 메타데이터를 활용해 데이터가 관련성 있고 안전하도록 보장하세요.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>